In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


C:\Users\Abhinesh Singh\AppData\Local\Temp\ipykernel_4768\2891487276.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\RAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv

# Load .env first
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
# Check key
print(os.getenv("GROQ_API_KEY"))


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


In [3]:
##  Step 1 : Load the txt file
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [4]:
# Step 2  FAISS vectorStore with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name ="all-MiniLM-L6-v2")
vectorStore = FAISS.from_documents(chunks,embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8535.28it/s]


In [5]:
## Step 3  Create a MMR Retriever
retriever = vectorStore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

In [6]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")

llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

In [7]:
# Step 5 RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
rag_chain= create_retrieval_chain(retriever=retriever,combine_docs_chain=document_chain)

In [8]:
# Step 6: Query

query = {
    "input": "How does LangChain support agents and memory?"
}

response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain supports agents by allowing them to interact with external APIs and databases, and to use tools like calculators, search APIs, or custom functions based on the instructions they receive. Additionally, LangChain supports memory in two ways: 

1. Conversational memory using ConversationBufferMemory, which helps models retain previous interactions, making multi-turn conversations more coherent.
2. Summarization memory with ConversationSummaryMemory. 

This enables LangChain agents to decide which tool to call and in what order during a task, enhancing the capabilities of LLM-powered applications.


In [9]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='41526005-67c1-4c2e-bb66-36b7f7a23698', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='2425d15c-e862-4616-b473-600d86b7ef21', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='df169469-b078-480b-967a-58be896d68d3', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri